# Qwen2.5-1.5B — treinamento e comparação reais

Este caderno lê evidências já concluídas; não simula respostas nem repete treino. Dados sintéticos e revisão técnica por IA, sem confirmação humana independente. As fontes e limitações estão no relatório da versão.

In [1]:
import json
from pathlib import Path
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists())
RUN=ROOT/'experimentos/v5-1.5b'
def load(path):
    return json.loads((RUN/path).read_text())
from citizentech_ia.datasets import verify_freeze
from citizentech_ia.v5.inference import SYSTEM
verify_freeze(RUN,system_prompt=SYSTEM)
print(json.dumps(load('dados/manifest.json'),ensure_ascii=False,indent=2))

{
  "version": "v5-1.5b",
  "counts": {
    "train": 400,
    "valid": 80,
    "test": 30
  },
  "train_valid_origin": "v3 data unchanged; previous observed validation reused, declared",
  "test_origin": "30 new synthetic compositional cases; no paraphrase replication; created before1.5B training/generation",
  "families_per_split": {
    "train": 50,
    "valid": 20,
    "test": 30
  },
  "independent_human_review": false,
  "limitations": [
    "shared domain rules and author/reviewer AI assistance",
    "new wording/compositions do not prove semantic independence or field generalization",
    "all target causal hypotheses empty",
    "legacy120 cases previously observed and stored as regression only"
  ],
  "hashes": {
    "dados/train.jsonl": "e024523512b7b090b6f6cf913b4a95ebe96db76fc7a42ef770b216db365b658b",
    "dados/valid.jsonl": "64aa22cf77b68b126358205e827a820517d1842fc3c9e0112f1c94b7026e88b8",
    "avaliacao/test.jsonl": "367fd3826b962a95f7dba1832bc2c069fdd8615b7c0885d01b706

## Treinamento e seleção

LoRA FP32 protegido; base FP16 preservada. Seleção pelos critérios predefinidos da validação64, antes das respostas do teste30. Validação bruta80 preservada. Pesos finais exatos, sem escolha por melhor loss.

In [2]:
selection=load('configs/selected.json')
print(json.dumps(selection,ensure_ascii=False,indent=2))
print(json.dumps(load('docs/smoke-e-estimativa.json'),ensure_ascii=False,indent=2))

{
  "selected_at": "2026-11-17T21:16:25.257044-03:00",
  "candidate": "r1",
  "checkpoint": "exact final weights of the selected candidate; 400 or 800 microsteps as recorded",
  "adapter_path": "adapters/selected/final",
  "adapter_sha": "c905fee58cb4c0c44baeb9ac14dd9e9b7e412f280b3024c825441d018d573dce",
  "plan_sha": "b319153a57d3f52256ee446e0491583466b3a396739f54a3e4c4b173ec17d0a1",
  "second_candidate_condition_met": true,
  "criterion": "clean validation: pure status, valid schema/references, schema, latency; lexicographic",
  "candidates": {
    "r1": {
      "clean_metrics": {
        "analysis_set": "inherited64 validation cases without previously identified overlap",
        "plan_sha": "b319153a57d3f52256ee446e0491583466b3a396739f54a3e4c4b173ec17d0a1",
        "mode": "rag_finetuning",
        "split": "valid",
        "n": 64,
        "n_full": 80,
        "n_excluded": 16,
        "json_syntax_valid_rate": 1.0,
        "schema_valid_rate": 1.0,
        "valid_json_and_refs_r

## Mesmas30 perguntas, três configurações

A:sem contexto. B:RAG simples. C:RAG+LoRA. B/C recebem as mesmas mensagens e trechos. Status puro e esquema não são acerto semântico.

In [3]:
for mode in ['final-A','final-B','final-C']:
    m=load(f'avaliacao/evidencias/{mode}/metrics.json')
    keys=['n','schema_valid_rate','valid_json_and_refs_rate','status_only_match_rate','median_seconds','p95_seconds','retrieval_respondible']
    print(mode,json.dumps({k:m[k] for k in keys},ensure_ascii=False))

final-A {"n": 30, "schema_valid_rate": 0.26666666666666666, "valid_json_and_refs_rate": 0.26666666666666666, "status_only_match_rate": 0.1, "median_seconds": 1.31454016600037, "p95_seconds": 1.6753758271966945, "retrieval_respondible": {"recall@1": 0.0, "recall@3": 0.0}}
final-B {"n": 30, "schema_valid_rate": 0.06666666666666667, "valid_json_and_refs_rate": 0.06666666666666667, "status_only_match_rate": 0.0, "median_seconds": 1.4985765835008351, "p95_seconds": 2.286958164398311, "retrieval_respondible": {"recall@1": 0.5, "recall@3": 0.9}}
final-C {"n": 30, "schema_valid_rate": 1.0, "valid_json_and_refs_rate": 1.0, "status_only_match_rate": 0.9333333333333333, "median_seconds": 1.6438996454999142, "p95_seconds": 1.7981474268974125, "retrieval_respondible": {"recall@1": 0.5, "recall@3": 0.9}}


## Revisão das respostas

Acerto, alucinação, fundamentação e recusa são julgamentos técnicos por IA. Incertos ficam no denominador por limites inferior/superior. Intervalos pareados por família são condicionais aos rótulos e ao conjunto sintético; não medem erro de anotação nem validade de campo.

In [4]:
m=load('revisao/test/metrics.json')
for mode,s in m['modes'].items():
    print(mode,json.dumps({k:s[k] for k in ['n','semantic_correct','integral_accuracy','hallucination','refusal_adequate']},ensure_ascii=False))
print(json.dumps(m['comparisons'],ensure_ascii=False,indent=2))

final-A {"n": 30, "semantic_correct": {"n": 30, "positive": 14, "negative": 15, "uncertain": 1, "lower_rate": 0.4666666666666667, "upper_rate": 0.5}, "integral_accuracy": {"n": 30, "positive": 3, "negative": 27, "uncertain": 0, "lower_rate": 0.1, "upper_rate": 0.1}, "hallucination": {"n": 30, "positive": 7, "negative": 21, "uncertain": 2, "lower_rate": 0.23333333333333334, "upper_rate": 0.3}, "refusal_adequate": {"n": 30, "positive": 20, "negative": 10, "uncertain": 0, "lower_rate": 0.6666666666666666, "upper_rate": 0.6666666666666666}}
final-B {"n": 30, "semantic_correct": {"n": 30, "positive": 10, "negative": 19, "uncertain": 1, "lower_rate": 0.3333333333333333, "upper_rate": 0.36666666666666664}, "integral_accuracy": {"n": 30, "positive": 0, "negative": 30, "uncertain": 0, "lower_rate": 0.0, "upper_rate": 0.0}, "hallucination": {"n": 30, "positive": 12, "negative": 17, "uncertain": 1, "lower_rate": 0.4, "upper_rate": 0.43333333333333335}, "refusal_adequate": {"n": 30, "positive": 19

## Erros e POC

Casos ilustrativos não são um teste independente. HTTP200 valida transporte e estrutura, não ausência de erro. Não atribuir resultados históricos a este modelo.

In [5]:
print(json.dumps(load('avaliacao/cinco_erros.json'),ensure_ascii=False,indent=2))
print(json.dumps(load('docs/teste-poc-real.json'),ensure_ascii=False,indent=2))

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "chosen_after_review_for_error_analysis": true,
  "independent_human_review": false,
  "composition": "fourC examples (one definite grounding failure with uncertain semantic/hallucination), oneB example; not five definite hallucinations of C",
  "cases": [
    {
      "review_id": "6abfda30f82b",
      "case_id": "a0426ce8cadcfe46d21755b6",
      "mode": "final-C",
      "category": "respondivel",
      "input": {
        "pergunta": "Como registrar os dois estados sem tratá-los automaticamente como conflito?",
        "municipio": null,
        "observado_em": null,
        "simulada": true,
        "relato": "Na mesma rua, o pontoA foi visto aceso e o pontoB apagado às22h. São luminárias diferentes.",
        "sensores": [],
        "observacoes_visuais": []
      },
      "context": [
        {
          "chunk_id": "conflito:1",
          "document_id": "conflito",
          "title": "Evidências contraditórias",
          "text": "Observ

## Limites

Corpus com dez procedimentos autorais e duas sínteses institucionais atribuídas. 400/80 herdados da v3; o teste30 foi criado antes destas gerações, mas com autoria/revisão relacionadas e regras de domínio compartilhadas. Não é conjunto externo independente. Todos os alvos têm hipóteses causais vazias: a tarefa é triagem, não diagnóstico elétrico. Teste antigo120 é regressão já observada. Publicação, integração real do grupo, integrantes/RMs/território, vídeo e revisão humana continuam pendentes. Consulte o relatório para erros remanescentes e reprodução.